# 무릎 통과에서 바 접촉까지 원판은 몇 px 올라갔을까?

검증 6명의 영상에서 직접 표시한 허브와 원판 베이스라인의 박스 중심을 비교합니다. 36번 실험에서 수동 허브를 확인한 같은 영상입니다. 1차 판독의 `transition`은 무릎 통과, `drive`는 바와 몸의 접촉입니다. 기본 입력 640과 이전 비교 입력 800의 저장된 원본 추론 결과를 사용합니다.

In [ ]:
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT
from wlcv.evaluation.phase_hub_rise import phase_rises, plot_endpoints, plot_rise_error

reference = ROOT / "outputs/plate_resolution/2026-11-24/paired_frames.csv"
pairs = pd.read_csv(reference)
rises = phase_rises(pairs)
output = ROOT / "outputs/phase_hub_rise" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
output.mkdir(parents=True)
rises.to_csv(output / "phase_rises.csv", index=False)

## 먼저 b02의 원본 두 장 확인하기

금색 원은 수동 허브, 하늘색 ×와 자홍색 +는 모델의 박스 중심입니다. 화면 아래쪽으로 y좌표가 증가합니다. 다른 영상을 확인하려면 아래 영상 ID를 바꿉니다.

In [ ]:
video = "b02_sn_h_02"
fig = plot_endpoints(pairs, rises, video)
fig.savefig(output / f"{video}_endpoints.png", dpi=150)
display(fig)
plt.close(fig)

## 수동 상승량과 모델 상승량 비교

각 상승량은 무릎 통과 y에서 접촉 y를 뺀 값입니다. 표의 마지막 열은 모델 상승량에서 수동 상승량을 뺀 값입니다. 음수면 모델이 덜 올라갔다고 측정한 것입니다. 판독 불가 또는 모델 출력 누락이 있는 구간의 오차는 빈칸으로 남습니다.

In [ ]:
cols = ["person_id", "imgsz", "knee_frame", "contact_frame", "manual_rise_px",
        "model_rise_px", "rise_error_px"]
display(rises[cols].round(2))
print("비교한 구간 수 (입력 크기별)")
display(rises.groupby("imgsz").rise_error_px.agg(["size", "count"]))

## 끝점 오차가 상승량에 얼마나 남는지 확인

끝점 오차는 모델 중심 y에서 수동 허브 y를 뺀 값입니다. 같은 방향으로 비슷하게 치우치면 두 점 사이의 상승량에서 상쇄됩니다. 반대 방향이면 오차가 더해집니다. 두 끝점에 같은 방향의 오차가 있어도 크기가 다르면 차이가 남습니다. 서로 다른 영상의 픽셀은 촬영 배율에 따라 실제 길이가 다릅니다.

In [ ]:
display(rises[["person_id", "imgsz", "knee_source", "contact_source", "knee_bias_px",
               "contact_bias_px", "rise_error_px"]].round(2))
fig = plot_rise_error(rises)
fig.savefig(output / "rise_error.png", dpi=150)
display(fig)
plt.close(fig)
print(output)